# Deep Learning for Biology

**Tier 3 -- Applied Bioinformatics**

Deep learning has driven breakthroughs across biology: AlphaFold solved the protein structure prediction problem, transformer-based protein language models capture evolutionary information from raw sequences, and variational autoencoders reveal latent structure in single-cell data. This notebook builds from classical ML foundations (covered in Module 07) and introduces the deep learning architectures most relevant to bioinformatics.

**Prerequisites:** Module 07 (Machine Learning for Biology), NumPy, pandas, matplotlib  
**Libraries:** `torch` (PyTorch), `numpy`, `pandas`, `matplotlib`

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Molecular Modeling and Docking](../../Tier_3_Applied_Bioinformatics/09_Molecular_Modeling_and_Docking/01_molecular_modeling_and_docking.ipynb) | [Next: Clinical Genomics: From Variants to Patient Care →](../../Tier_3_Applied_Bioinformatics/11_Clinical_Genomics/01_clinical_genomics.ipynb)

---

In [ ]:
# Installation (run once)
# For CPU-only PyTorch (sufficient for this notebook):
#   pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
# For GPU-enabled PyTorch (requires CUDA):
#   pip install torch torchvision
# On Google Colab, PyTorch is pre-installed.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

---
## 1. From Classical ML to Deep Learning

### When Do You Need Deep Learning?

Classical ML (random forests, SVMs, logistic regression) remains the right choice for many biological problems, especially when:
- You have **tabular features** (gene expression, k-mer counts, physicochemical properties)
- Datasets are **small** (hundreds to low thousands of samples)
- **Interpretability** is critical

Deep learning excels when:
- Input is **raw sequence/structure/image** data (no hand-crafted features needed)
- Datasets are **large** (tens of thousands or more samples)
- The problem has **hierarchical patterns** (motifs within motifs)
- You can leverage **pre-trained models** via transfer learning

| Criterion | Classical ML | Deep Learning |
|-----------|-------------|---------------|
| Sample size needed | 100s--1000s | 1000s--millions |
| Feature engineering | Manual (domain knowledge) | Learned automatically |
| Interpretability | High (feature importance) | Lower (requires special methods) |
| Raw sequences/images | Needs preprocessing | Native input |
| Training time | Minutes | Hours--days |
| Hardware | CPU | GPU recommended |

### Neural Network Fundamentals

A neural network is a series of **layers** that transform input data through learnable linear transformations followed by nonlinear **activation functions**.

**The Perceptron (single neuron):**

$$y = \sigma(\mathbf{w} \cdot \mathbf{x} + b)$$

where $\mathbf{w}$ are weights, $b$ is a bias, and $\sigma$ is an activation function.

**Common activation functions:**
- **Sigmoid:** $\sigma(z) = \frac{1}{1 + e^{-z}}$ -- outputs in (0, 1), used for binary classification output
- **ReLU:** $\sigma(z) = \max(0, z)$ -- default choice for hidden layers, avoids vanishing gradients
- **Tanh:** $\sigma(z) = \tanh(z)$ -- outputs in (-1, 1), centered around zero

**Backpropagation** computes the gradient of the loss with respect to every weight using the chain rule, allowing gradient descent to update the weights.

**Universal Approximation Theorem (intuition):** A neural network with a single hidden layer containing enough neurons can approximate any continuous function to arbitrary precision. In practice, *deeper* networks learn hierarchical representations more efficiently than very wide shallow ones.

In [ ]:
# Visualize activation functions
z = np.linspace(-5, 5, 200)

sigmoid = 1 / (1 + np.exp(-z))
relu = np.maximum(0, z)
tanh = np.tanh(z)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, vals, name in zip(axes, [sigmoid, relu, tanh], ['Sigmoid', 'ReLU', 'Tanh']):
    ax.plot(z, vals, linewidth=2)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(name)
    ax.set_xlabel('z')
    ax.set_ylabel('activation(z)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. Building Neural Networks with PyTorch

### Tensors and Autograd

PyTorch tensors are like NumPy arrays but can track gradients for automatic differentiation.

In [ ]:
# Tensors: creating and basic operations
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2 + 2 * x + 1  # elementwise quadratic
loss = y.sum()

# Backpropagation: compute dy/dx
loss.backward()

print(f"x     = {x.data}")
print(f"y     = {y.data}")
print(f"dy/dx = {x.grad}")  # dy/dx = 2x + 2
print(f"Expected: {2 * x.data + 2}")

In [ ]:
# Converting between NumPy and PyTorch
np_array = np.array([[1, 2], [3, 4]], dtype=np.float32)
tensor = torch.from_numpy(np_array)
back_to_np = tensor.numpy()

print(f"NumPy shape: {np_array.shape}, dtype: {np_array.dtype}")
print(f"Tensor shape: {tensor.shape}, dtype: {tensor.dtype}")
print(f"Shares memory: {np.shares_memory(np_array, back_to_np)}")

### Defining Models with nn.Module

Every PyTorch model inherits from `nn.Module`. You define the layers in `__init__` and the forward pass in `forward`. Backpropagation through the layers is handled automatically.

In [ ]:
class SimpleClassifier(nn.Module):
    """A feedforward network for binary classification."""

    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

# Inspect the model
model = SimpleClassifier(input_dim=16)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

### Case Study: Promoter vs Non-Promoter Classification

We build a binary classifier using **dinucleotide frequencies** as features. This parallels the classical ML approach from Module 07 but uses a neural network.

In [ ]:
# Generate synthetic promoter / non-promoter sequences
def generate_sequences(n_per_class=500, seq_length=200):
    sequences, labels = [], []
    for _ in range(n_per_class):
        # Promoter: enriched in CG dinucleotides, TATA-like motifs
        seq = []
        for j in range(seq_length):
            if np.random.random() < 0.15:
                seq.append(np.random.choice(['C', 'G']))
            elif 80 <= j <= 90 and np.random.random() < 0.6:
                seq.extend(list('TATAAA'[:min(6, seq_length - j)]))
            else:
                seq.append(np.random.choice(['A', 'T', 'G', 'C'], p=[0.2, 0.2, 0.35, 0.25]))
        sequences.append(''.join(seq[:seq_length]))
        labels.append(1)

        # Non-promoter: uniform composition
        seq = ''.join(np.random.choice(['A', 'T', 'G', 'C'], size=seq_length))
        sequences.append(seq)
        labels.append(0)
    return sequences, np.array(labels)


def dinucleotide_features(sequences):
    """Extract 16 dinucleotide frequencies from each sequence."""
    nucleotides = ['A', 'C', 'G', 'T']
    dinucs = [a + b for a in nucleotides for b in nucleotides]
    features = np.zeros((len(sequences), 16))
    for i, seq in enumerate(sequences):
        total = len(seq) - 1
        for j in range(total):
            dinuc = seq[j:j+2]
            if dinuc in dinucs:
                features[i, dinucs.index(dinuc)] += 1
        features[i] /= total  # normalize to frequencies
    return features, dinucs


sequences, labels = generate_sequences(500)
X, feature_names = dinucleotide_features(sequences)
print(f"Dataset: {X.shape[0]} sequences, {X.shape[1]} features")
print(f"Labels: {np.sum(labels == 1)} promoters, {np.sum(labels == 0)} non-promoters")

In [ ]:
# The standard PyTorch training loop
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42, stratify=labels
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

# DataLoader for mini-batch training
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Model, loss, optimizer
model = SimpleClassifier(input_dim=16, hidden_dim=64).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
train_losses = []
for epoch in range(100):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()           # 1. Reset gradients
        predictions = model(X_batch)    # 2. Forward pass
        loss = criterion(predictions, y_batch)  # 3. Compute loss
        loss.backward()                 # 4. Backpropagation
        optimizer.step()                # 5. Update weights

        epoch_loss += loss.item() * X_batch.size(0)

    train_losses.append(epoch_loss / len(train_dataset))

# Evaluate
model.eval()
with torch.no_grad():
    test_preds = model(X_test_t.to(device)).cpu()
    test_acc = ((test_preds > 0.5).float() == y_test_t).float().mean()

print(f"Final training loss: {train_losses[-1]:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

In [ ]:
# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(train_losses, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('Training Loss -- Promoter Classifier')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Convolutional Neural Networks for Sequences

### One-Hot Encoding of Biological Sequences

Instead of extracting hand-crafted features (k-mer frequencies), we can feed **raw sequences** into a CNN. Each position becomes a 4-dimensional one-hot vector (for DNA) or a 20-dimensional vector (for proteins).

```
Sequence: A  C  G  T  A
A:        1  0  0  0  1
C:        0  1  0  0  0
G:        0  0  1  0  0
T:        0  0  0  1  0
```

A 1D convolution slides a **filter** (kernel) across the sequence. Filters learn to detect motifs -- just like k-mers, but the network discovers the relevant patterns itself.

In [ ]:
def one_hot_encode(sequences, alphabet='ACGT'):
    """One-hot encode a list of sequences.

    Returns tensor of shape (N, C, L) where C = len(alphabet), L = seq_length.
    This matches PyTorch Conv1d expected input: (batch, channels, length).
    """
    mapping = {c: i for i, c in enumerate(alphabet)}
    n = len(sequences)
    seq_len = len(sequences[0])
    encoded = np.zeros((n, len(alphabet), seq_len), dtype=np.float32)
    for i, seq in enumerate(sequences):
        for j, char in enumerate(seq[:seq_len]):
            if char in mapping:
                encoded[i, mapping[char], j] = 1.0
    return torch.FloatTensor(encoded)


# Encode our sequences
X_encoded = one_hot_encode(sequences)
print(f"Encoded shape: {X_encoded.shape}  (samples, channels=4, length=200)")

# Visualize one-hot encoding for a short segment
fig, ax = plt.subplots(figsize=(12, 2))
segment = X_encoded[0, :, :30].numpy()
ax.imshow(segment, aspect='auto', cmap='Blues', interpolation='nearest')
ax.set_yticks(range(4))
ax.set_yticklabels(['A', 'C', 'G', 'T'])
ax.set_xlabel('Position')
ax.set_title('One-hot encoding (first 30 bp of a promoter)')
plt.tight_layout()
plt.show()

### 1D CNN Architecture for TF Binding Site Prediction

The architecture follows a proven pattern for biological sequence classification:

**Conv1D** (detect motifs) --> **ReLU** (nonlinearity) --> **MaxPool** (position invariance) --> **Dense** (classify)

Multiple convolutional layers can capture motif combinations and spacing patterns.

In [ ]:
class SequenceCNN(nn.Module):
    """1D CNN for DNA sequence classification.

    Architecture:
        Conv1d(4, 32, kernel=8) -> ReLU -> MaxPool(4)
        Conv1d(32, 64, kernel=6) -> ReLU -> MaxPool(4)
        Global average pooling -> Dense(64) -> ReLU -> Dense(1) -> Sigmoid
    """

    def __init__(self, seq_length=200):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv1d(in_channels=4, out_channels=32, kernel_size=8, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=6, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x shape: (batch, 4, seq_length)
        x = self.conv_layers(x)        # (batch, 64, reduced_length)
        x = x.mean(dim=2)              # global average pooling -> (batch, 64)
        x = self.classifier(x)         # (batch, 1)
        return x


cnn_model = SequenceCNN(seq_length=200)
print(cnn_model)
print(f"\nParameters: {sum(p.numel() for p in cnn_model.parameters()):,}")

In [ ]:
# Train the CNN on raw sequences
y_tensor = torch.FloatTensor(labels).unsqueeze(1)

# Split
idx = np.arange(len(sequences))
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42, stratify=labels)

X_train_cnn = X_encoded[train_idx]
y_train_cnn = y_tensor[train_idx]
X_test_cnn = X_encoded[test_idx]
y_test_cnn = y_tensor[test_idx]

train_ds = TensorDataset(X_train_cnn, y_train_cnn)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

cnn_model = SequenceCNN().to(device)
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)
criterion = nn.BCELoss()

cnn_losses = []
for epoch in range(50):
    cnn_model.train()
    epoch_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = cnn_model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    cnn_losses.append(epoch_loss / len(train_ds))

cnn_model.eval()
with torch.no_grad():
    test_pred = cnn_model(X_test_cnn.to(device)).cpu()
    cnn_acc = ((test_pred > 0.5).float() == y_test_cnn).float().mean()

print(f"CNN Test accuracy: {cnn_acc:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(cnn_losses, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CNN Training Loss -- Sequence Classification')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### What Do the Filters Learn?

CNN filters in the first layer learn to recognize short sequence motifs. We can extract and visualize them as position weight matrices (PWMs), similar to known transcription factor binding motifs.

In [ ]:
# Extract first-layer convolutional filters as motifs
filters = cnn_model.conv_layers[0].weight.data.cpu().numpy()  # (32, 4, 8)

fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for i, ax in enumerate(axes.flat):
    # Show filter as a heatmap (like a PWM)
    filt = filters[i]  # shape (4, 8)
    ax.imshow(filt, aspect='auto', cmap='RdBu_r', interpolation='nearest',
              vmin=-filt.max(), vmax=filt.max())
    ax.set_yticks(range(4))
    ax.set_yticklabels(['A', 'C', 'G', 'T'])
    ax.set_title(f'Filter {i}')
    ax.set_xlabel('Position')

plt.suptitle('Learned CNN Filters (first 8 of 32)', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Recurrent Neural Networks

### Sequence Modeling with RNNs, LSTMs, and GRUs

Recurrent neural networks process sequences **one element at a time**, maintaining a hidden state that captures context from previous positions. This makes them naturally suited for biological sequences.

**Vanilla RNN:**
$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$$

Problem: vanilla RNNs suffer from **vanishing gradients** -- they struggle to learn long-range dependencies.

**LSTM (Long Short-Term Memory)** solves this with gating mechanisms:
- **Forget gate:** what to discard from the cell state
- **Input gate:** what new information to store
- **Output gate:** what to output from the cell state

**GRU (Gated Recurrent Unit)** is a simplified LSTM with two gates (reset and update), often performing comparably with fewer parameters.

In [ ]:
class SequenceLSTM(nn.Module):
    """Bidirectional LSTM for DNA sequence classification."""

    def __init__(self, input_dim=4, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        # Bidirectional doubles the hidden dim
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: (batch, channels=4, seq_length) -> need (batch, seq_length, 4)
        x = x.transpose(1, 2)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # Use the last hidden state from both directions
        # h_n shape: (num_layers * 2, batch, hidden_dim)
        forward_h = h_n[-2]   # last layer, forward
        backward_h = h_n[-1]  # last layer, backward
        combined = torch.cat([forward_h, backward_h], dim=1)
        return self.classifier(combined)


lstm_model = SequenceLSTM()
print(lstm_model)
print(f"\nParameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

In [ ]:
# Train the LSTM (fewer epochs due to slower training)
lstm_model = SequenceLSTM().to(device)
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

lstm_losses = []
for epoch in range(30):
    lstm_model.train()
    epoch_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = lstm_model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    lstm_losses.append(epoch_loss / len(train_ds))

lstm_model.eval()
with torch.no_grad():
    test_pred = lstm_model(X_test_cnn.to(device)).cpu()
    lstm_acc = ((test_pred > 0.5).float() == y_test_cnn).float().mean()

print(f"LSTM Test accuracy: {lstm_acc:.4f}")

### RNNs for Protein Secondary Structure Prediction

Protein secondary structure prediction (helix/sheet/coil for each residue) is a classic **sequence-to-sequence** task where RNNs shine. Each amino acid position gets a label:

| Label | Structure | Description |
|-------|-----------|-------------|
| H | Alpha-helix | Hydrogen bonds i -> i+4 |
| E | Beta-sheet | Extended, forms parallel/antiparallel sheets |
| C | Coil | Everything else |

Modern methods like NetSurfP-3.0 use transformer-based protein language model embeddings as input, achieving ~87% per-residue accuracy (Q3).

In [ ]:
class SecondaryStructureRNN(nn.Module):
    """Toy model: predict 3-state secondary structure from amino acid sequence.

    In practice, you would use pre-trained protein language model embeddings
    (e.g., ESM-2) instead of one-hot amino acid encoding.
    """

    def __init__(self, n_amino_acids=20, hidden_dim=64, n_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(n_amino_acids, 32)
        self.lstm = nn.LSTM(
            input_size=32, hidden_size=hidden_dim,
            num_layers=2, batch_first=True, bidirectional=True
        )
        self.output = nn.Linear(hidden_dim * 2, n_classes)  # per-residue prediction

    def forward(self, x):
        # x: (batch, seq_length) integer-encoded amino acids
        x = self.embedding(x)           # (batch, seq_length, 32)
        x, _ = self.lstm(x)             # (batch, seq_length, hidden*2)
        x = self.output(x)              # (batch, seq_length, 3)
        return x

# Demonstrate with a synthetic example
ss_model = SecondaryStructureRNN()
dummy_seq = torch.randint(0, 20, (4, 100))  # batch of 4 sequences, length 100
output = ss_model(dummy_seq)
print(f"Input shape:  {dummy_seq.shape}  (batch, sequence_length)")
print(f"Output shape: {output.shape}  (batch, sequence_length, 3_classes)")
print(f"Predicted structure for first residue of first sequence:")
print(f"  Logits: {output[0, 0].detach().numpy().round(3)}")
print(f"  Predicted: {['H', 'E', 'C'][output[0, 0].argmax().item()]}")

---
## 5. Attention and Transformers

### The Self-Attention Mechanism

Transformers replaced RNNs by processing all positions **in parallel** using self-attention. Each position attends to every other position, learning which parts of the sequence are relevant to each other.

**Scaled dot-product attention:**

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

where Q (queries), K (keys), V (values) are linear projections of the input. The $\sqrt{d_k}$ scaling prevents the dot products from becoming too large.

**Multi-head attention** runs several attention computations in parallel, allowing the model to attend to information from different representation subspaces.

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    """Compute scaled dot-product attention from scratch."""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    attention_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)
    return output, attention_weights


# Demonstration: a short "protein" sequence
seq_length = 8
d_model = 16

# Random embeddings for 8 positions
x = torch.randn(1, seq_length, d_model)

# Linear projections
W_Q = nn.Linear(d_model, d_model, bias=False)
W_K = nn.Linear(d_model, d_model, bias=False)
W_V = nn.Linear(d_model, d_model, bias=False)

Q, K, V = W_Q(x), W_K(x), W_V(x)
output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"Input shape:           {x.shape}")
print(f"Attention weights:     {attn_weights.shape}")
print(f"Output shape:          {output.shape}")

# Visualize attention
plt.figure(figsize=(6, 5))
residues = ['M', 'A', 'K', 'L', 'V', 'E', 'G', 'S']
plt.imshow(attn_weights[0].detach().numpy(), cmap='viridis')
plt.xticks(range(8), residues)
plt.yticks(range(8), residues)
plt.xlabel('Key (attending to)')
plt.ylabel('Query (attending from)')
plt.title('Self-Attention Weights')
plt.colorbar(label='Attention weight')
plt.tight_layout()
plt.show()

### Transformers in Biology

Transformers have revolutionized computational biology:

**Protein language models:**
- **ESM-2** (Meta AI): Trained on millions of protein sequences. Embeddings capture structure, function, and evolutionary relationships. The attention heads learn to identify residue contacts.
- **ProtTrans** (Rostlab): Family of transformer models (ProtBERT, ProtT5) for protein sequences.

**Structure prediction:**
- **AlphaFold2** (DeepMind): Uses a novel "Evoformer" attention architecture over MSAs and pair representations to predict 3D protein structures with atomic accuracy.
- **ESMFold**: Predicts structure directly from a single sequence using ESM-2 embeddings -- no MSA needed.

**Genomics:**
- **DNABERT**: BERT-style pre-training on DNA k-mers for regulatory element prediction.
- **Enformer** (DeepMind): Predicts gene expression from DNA sequence using attention over 200 kb of genomic context.
- **Nucleotide Transformer** (InstaDeep): Foundation model for genomics tasks.

In [ ]:
class TransformerClassifier(nn.Module):
    """Transformer encoder for sequence classification."""

    def __init__(self, input_dim=4, d_model=64, nhead=4, num_layers=2, max_len=200):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)

        # Positional encoding (learnable)
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: (batch, 4, seq_length) -> (batch, seq_length, 4)
        x = x.transpose(1, 2)
        x = self.input_proj(x) + self.pos_embedding[:, :x.size(1), :]
        x = self.transformer(x)         # (batch, seq_length, d_model)
        x = x.mean(dim=1)               # global average pooling
        return self.classifier(x)


transformer_model = TransformerClassifier().to(device)
print(f"Parameters: {sum(p.numel() for p in transformer_model.parameters()):,}")

# Quick training
optimizer = optim.Adam(transformer_model.parameters(), lr=0.0005)
for epoch in range(30):
    transformer_model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(transformer_model(xb), yb)
        loss.backward()
        optimizer.step()

transformer_model.eval()
with torch.no_grad():
    test_pred = transformer_model(X_test_cnn.to(device)).cpu()
    trans_acc = ((test_pred > 0.5).float() == y_test_cnn).float().mean()

print(f"Transformer Test accuracy: {trans_acc:.4f}")

In [ ]:
# Compare all three architectures
results = pd.DataFrame({
    'Model': ['Feedforward (dinuc features)', 'CNN (raw sequence)', 'LSTM (raw sequence)', 'Transformer (raw sequence)'],
    'Test Accuracy': [test_acc.item(), cnn_acc.item(), lstm_acc.item(), trans_acc.item()],
    'Parameters': [
        sum(p.numel() for p in SimpleClassifier(16, 64).parameters()),
        sum(p.numel() for p in SequenceCNN().parameters()),
        sum(p.numel() for p in SequenceLSTM().parameters()),
        sum(p.numel() for p in TransformerClassifier().parameters())
    ]
})
results['Parameters'] = results['Parameters'].apply(lambda x: f"{x:,}")
print(results.to_string(index=False))

---
## 6. Autoencoders and Representation Learning

### Autoencoder Basics

An autoencoder compresses input into a low-dimensional **latent space** (encoder) and reconstructs the original input (decoder). The bottleneck forces the model to learn the most informative features.

```
Input (high-dim) -> Encoder -> Latent (low-dim) -> Decoder -> Reconstruction
```

### Variational Autoencoder (VAE)

A VAE learns a *probabilistic* latent space, where each input maps to a distribution (mean + variance). This enables:
- **Smooth interpolation** between data points
- **Generation** of new samples by sampling from the latent space

The VAE loss combines reconstruction loss + a KL divergence term that regularizes the latent space:

$$\mathcal{L} = \text{Reconstruction Loss} + \beta \cdot D_{KL}(q(z|x) \| p(z))$$

In [ ]:
class VAE(nn.Module):
    """Variational Autoencoder for gene expression data."""

    def __init__(self, input_dim, hidden_dim=128, latent_dim=10):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden_dim // 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        """Reparameterization trick: sample z = mu + sigma * epsilon."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    """VAE loss = MSE reconstruction + beta * KL divergence."""
    recon_loss = nn.functional.mse_loss(recon_x, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl_loss

In [ ]:
# Application: single-cell gene expression denoising
# Simulate 3 cell types with 500 genes
n_genes = 500
n_cells_per_type = 200
cell_types = []
expression_data = []

for ct, (start, end) in enumerate([(0, 50), (50, 100), (100, 150)]):
    for _ in range(n_cells_per_type):
        expr = np.random.exponential(0.5, n_genes).astype(np.float32)
        expr[start:end] += np.random.exponential(2.0, end - start)  # upregulated genes
        # Add dropout noise (common in scRNA-seq)
        dropout_mask = np.random.binomial(1, 0.3, n_genes)  # 30% dropout
        expr *= dropout_mask
        expression_data.append(expr)
        cell_types.append(ct)

X_sc = np.array(expression_data)
y_sc = np.array(cell_types)

# Log-transform (standard for scRNA-seq)
X_sc = np.log1p(X_sc)

print(f"Single-cell data: {X_sc.shape[0]} cells, {X_sc.shape[1]} genes")
print(f"Cell types: {np.bincount(y_sc)}")

In [ ]:
# Train the VAE
X_sc_tensor = torch.FloatTensor(X_sc)
sc_dataset = TensorDataset(X_sc_tensor)
sc_loader = DataLoader(sc_dataset, batch_size=64, shuffle=True)

vae = VAE(input_dim=n_genes, hidden_dim=128, latent_dim=10).to(device)
optimizer = optim.Adam(vae.parameters(), lr=0.001)

vae_losses = []
for epoch in range(100):
    vae.train()
    total_loss = 0
    for (batch,) in sc_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = vae(batch)
        loss = vae_loss(recon, batch, mu, logvar, beta=0.5)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    vae_losses.append(total_loss / len(X_sc))

print(f"Final VAE loss: {vae_losses[-1]:.2f}")

In [ ]:
# Visualize the learned latent space
vae.eval()
with torch.no_grad():
    mu_all, _ = vae.encode(X_sc_tensor.to(device))
    latent = mu_all.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA of raw data
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_sc)

colors = ['#e74c3c', '#3498db', '#2ecc71']
cell_type_names = ['Type A', 'Type B', 'Type C']
for ct in range(3):
    mask = y_sc == ct
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors[ct], label=cell_type_names[ct],
                    alpha=0.5, s=15)
    axes[1].scatter(latent[mask, 0], latent[mask, 1], c=colors[ct], label=cell_type_names[ct],
                    alpha=0.5, s=15)

axes[0].set_title('PCA of Raw Expression (noisy)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend()

axes[1].set_title('VAE Latent Space (denoised)')
axes[1].set_xlabel('Latent dim 1')
axes[1].set_ylabel('Latent dim 2')
axes[1].legend()

plt.tight_layout()
plt.show()

### Application: Molecule Generation

VAEs can learn latent representations of molecular structures (e.g., SMILES strings or molecular graphs). Sampling from the latent space generates novel molecules with desired properties. This is a core technique in **de novo drug design**.

Tools like:
- **Junction Tree VAE** (MIT): generates valid molecules by operating on molecular substructures
- **MolGAN**: combines GANs with reinforcement learning for molecular generation
- **REINVENT** (AstraZeneca): uses RNNs with reinforcement learning for focused molecular generation

---
## 7. Practical Considerations

### Overfitting in Biological Data

Biology often has the **small n, large p** problem: few samples but many features (genes, variants, etc.). This makes deep learning prone to overfitting.

In [ ]:
# Demonstration: overfitting on a small dataset
# Use only 100 samples (50 per class) with the CNN
small_idx = np.concatenate([
    np.where(labels == 1)[0][:50],
    np.where(labels == 0)[0][:50]
])
np.random.shuffle(small_idx)

small_train_idx = small_idx[:80]
small_test_idx = small_idx[80:]

small_train_ds = TensorDataset(X_encoded[small_train_idx], y_tensor[small_train_idx])
small_train_dl = DataLoader(small_train_ds, batch_size=16, shuffle=True)

# Train without regularization
overfit_model = SequenceCNN().to(device)
optimizer = optim.Adam(overfit_model.parameters(), lr=0.001)

train_accs, test_accs = [], []
for epoch in range(100):
    overfit_model.train()
    for xb, yb in small_train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(overfit_model(xb), yb)
        loss.backward()
        optimizer.step()

    overfit_model.eval()
    with torch.no_grad():
        train_pred = overfit_model(X_encoded[small_train_idx].to(device)).cpu()
        test_pred = overfit_model(X_encoded[small_test_idx].to(device)).cpu()
        train_accs.append(((train_pred > 0.5).float() == y_tensor[small_train_idx]).float().mean().item())
        test_accs.append(((test_pred > 0.5).float() == y_tensor[small_test_idx]).float().mean().item())

plt.figure(figsize=(8, 4))
plt.plot(train_accs, label='Training accuracy', linewidth=2)
plt.plot(test_accs, label='Test accuracy', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Overfitting with Small Biological Dataset (n=80)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final train accuracy: {train_accs[-1]:.3f}")
print(f"Final test accuracy:  {test_accs[-1]:.3f}")

### Strategies Against Overfitting

| Strategy | How it works | When to use |
|----------|-------------|-------------|
| **Dropout** | Randomly zero out neurons during training | Always (standard regularization) |
| **Early stopping** | Stop training when validation loss increases | Always |
| **Weight decay** | L2 penalty on weights | Most cases |
| **Data augmentation** | Create synthetic training examples | When dataset is small |
| **Transfer learning** | Start from pre-trained model | When a relevant pre-trained model exists |
| **Batch normalization** | Normalize layer inputs | Deep networks |

In [ ]:
# Data augmentation for DNA sequences
def augment_dna_sequence(seq):
    """Augment a DNA sequence using biologically motivated transformations."""
    augmented = []

    # 1. Reverse complement (biologically equivalent for double-stranded DNA)
    complement = str.maketrans('ACGT', 'TGCA')
    augmented.append(seq[::-1].translate(complement))

    # 2. Random point mutations (simulates sequencing errors / SNPs)
    mutated = list(seq)
    for _ in range(3):  # 3 random mutations
        pos = np.random.randint(len(seq))
        mutated[pos] = np.random.choice(['A', 'C', 'G', 'T'])
    augmented.append(''.join(mutated))

    # 3. Random subsequence shift (sliding window)
    shift = np.random.randint(-10, 11)
    if shift > 0:
        shifted = 'A' * shift + seq[:-shift]
    elif shift < 0:
        shifted = seq[-shift:] + 'A' * (-shift)
    else:
        shifted = seq
    augmented.append(shifted)

    return augmented


# Example
original = sequences[0][:50]
augmented = augment_dna_sequence(original)
print(f"Original:           {original}")
print(f"Reverse complement: {augmented[0]}")
print(f"Point mutations:    {augmented[1]}")
print(f"Shifted:            {augmented[2]}")

### Transfer Learning with Pre-trained Biological Models

Instead of training from scratch, you can use embeddings from pre-trained models as input features. This is especially powerful when your dataset is small.

**Common pre-trained models for biology:**

| Model | Domain | Training data | Embedding dim | Usage |
|-------|--------|--------------|--------------|-------|
| ESM-2 (Meta) | Proteins | 250M sequences | 320--5120 | `pip install fair-esm` |
| ProtT5 (Rostlab) | Proteins | UniRef50 | 1024 | HuggingFace |
| DNABERT-2 | DNA | Multi-species genomes | 768 | HuggingFace |
| Nucleotide Transformer | DNA | 3200 species | 512--2560 | HuggingFace |
| scGPT | Single-cell | 33M cells | 512 | GitHub |

```python
# Example: using ESM-2 embeddings (pseudo-code)
import esm

model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()

data = [("protein1", "MKTLLILAVLCLG...")]
_, _, tokens = batch_converter(data)

with torch.no_grad():
    results = model(tokens, repr_layers=[6])
    embeddings = results["representations"][6]  # (1, seq_len, 320)

# Use embeddings as features for your downstream classifier
```

### Interpreting Neural Networks

Understanding *why* a model makes a prediction is critical in biology. Common approaches:

In [ ]:
# Saliency map: which positions matter most for the prediction?
def compute_saliency(model, x):
    """Compute input-gradient saliency for a single sequence."""
    x = x.unsqueeze(0).requires_grad_(True)  # add batch dimension
    model.eval()
    output = model(x.to(device))
    output.backward()
    # Saliency = absolute gradient summed over channels
    saliency = x.grad.data.abs().squeeze(0).sum(dim=0).cpu().numpy()
    return saliency


# Compute saliency for a promoter sequence
promoter_idx = np.where(labels == 1)[0][0]
saliency = compute_saliency(cnn_model, X_encoded[promoter_idx])

plt.figure(figsize=(14, 3))
plt.bar(range(len(saliency)), saliency, width=1.0, color='steelblue', alpha=0.7)
plt.xlabel('Sequence Position')
plt.ylabel('Saliency (|gradient|)')
plt.title('Saliency Map -- Which Positions Drive the Promoter Prediction?')
plt.axvspan(80, 90, alpha=0.2, color='red', label='Expected TATA region')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### GPU vs CPU and Google Colab Setup

**When do you need a GPU?**
- Training CNNs/LSTMs on >10,000 sequences: GPU provides 10--50x speedup
- Using pre-trained models (ESM-2, AlphaFold): GPU essential for inference at scale
- Small feedforward networks on tabular data: CPU is fine

**Google Colab setup:**
```python
# In Colab: Runtime -> Change runtime type -> GPU (T4)
# PyTorch is pre-installed with CUDA support
import torch
print(torch.cuda.is_available())        # Should be True
print(torch.cuda.get_device_name(0))    # e.g., 'Tesla T4'

# Move model and data to GPU
device = torch.device('cuda')
model = model.to(device)
x = x.to(device)
```

**Memory management tips:**
- Use `torch.no_grad()` during inference to save memory
- Reduce batch size if you get CUDA out-of-memory errors
- Use mixed precision (`torch.cuda.amp`) for large models
- Free GPU memory with `torch.cuda.empty_cache()` when switching models

---
## 8. Biological Applications Overview

### AlphaFold and Protein Structure Prediction

AlphaFold2 (2020) solved a 50-year grand challenge in biology. Key architectural innovations:

1. **Evoformer:** A novel attention-based module that jointly reasons over:
   - MSA representation (evolution across species)
   - Pair representation (residue-residue relationships)
2. **Structure module:** Iteratively refines 3D coordinates using invariant point attention
3. **Recycling:** Feeds predictions back through the network 3 times for refinement

**Impact:** The AlphaFold Protein Structure Database now contains predicted structures for >200 million proteins. AlphaFold3 (2024) extends to protein-ligand, protein-DNA, and protein-RNA complexes.

### DeepVariant for Variant Calling

Google's DeepVariant reframes variant calling as an **image classification** problem:
1. Pileup of aligned reads around a candidate variant site is encoded as an RGB image
2. An Inception-v3 CNN classifies each site as: homozygous reference, heterozygous, or homozygous variant
3. Achieves state-of-the-art accuracy across sequencing platforms (Illumina, PacBio, Nanopore)

### Drug Discovery with Deep Learning

| Stage | DL Application | Example Tools |
|-------|---------------|---------------|
| Target identification | Gene expression analysis, protein function prediction | DeepChem, scVI |
| Hit finding | Virtual screening, molecular property prediction | Chemprop, SchNet |
| Lead optimization | Molecule generation, ADMET prediction | REINVENT, MolBART |
| Clinical trials | Patient stratification, outcome prediction | DeepSurv |

### Single-Cell Analysis with Autoencoders

**scVI** (single-cell Variational Inference) is the leading deep learning framework for scRNA-seq:
- Learns a latent representation while modeling technical noise (library size, batch effects)
- Enables: batch correction, differential expression, imputation, and cell type annotation
- Built on PyTorch with the **scvi-tools** library

```python
# Example usage (pseudo-code)
import scvi
scvi.model.SCVI.setup_anndata(adata, batch_key="batch")
model = scvi.model.SCVI(adata, n_latent=10)
model.train(max_epochs=200)
latent = model.get_latent_representation()  # denoised, batch-corrected
```

---
## Summary: Architecture Selection Guide

| Problem | Input | Recommended Architecture |
|---------|-------|--------------------------|
| Classification from tabular features | k-mer freq, expression profiles | Feedforward NN (or classical ML) |
| Motif detection in sequences | Raw DNA/protein | 1D CNN |
| Long-range sequence dependencies | Full-length sequences | Transformer or Bi-LSTM |
| Sequence-to-sequence prediction | Residue-level labels | Bi-LSTM or Transformer encoder |
| Dimensionality reduction / denoising | High-dimensional profiles | VAE (autoencoder) |
| Generation of new molecules | SMILES / molecular graphs | VAE or GAN |
| Protein structure prediction | Sequence + MSA | Transformer (AlphaFold-like) |
| Any task with small data | Any | Transfer learning + fine-tuning |

---
## Exercises

### Exercise 1: Early Stopping Implementation

Implement **early stopping** for the CNN promoter classifier:

1. Split the training data into 80% train / 20% validation
2. Train for up to 200 epochs, tracking both training and validation loss
3. Stop training when the validation loss has not improved for 15 consecutive epochs (patience=15)
4. Restore the model weights from the epoch with the best validation loss
5. Plot training vs validation loss, marking the early stopping point

In [ ]:
# Solution: Exercise 1
import copy

# Split training data further into train/val
val_split = int(0.8 * len(train_idx))
es_train_idx = train_idx[:val_split]
es_val_idx = train_idx[val_split:]

es_train_ds = TensorDataset(X_encoded[es_train_idx], y_tensor[es_train_idx])
es_train_dl = DataLoader(es_train_ds, batch_size=32, shuffle=True)

X_val = X_encoded[es_val_idx].to(device)
y_val = y_tensor[es_val_idx].to(device)

es_model = SequenceCNN().to(device)
optimizer = optim.Adam(es_model.parameters(), lr=0.001)

best_val_loss = float('inf')
best_model_state = None
patience = 15
patience_counter = 0
es_train_losses, es_val_losses = [], []
stop_epoch = 200

for epoch in range(200):
    # Training
    es_model.train()
    epoch_loss = 0
    for xb, yb in es_train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(es_model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    es_train_losses.append(epoch_loss / len(es_train_ds))

    # Validation
    es_model.eval()
    with torch.no_grad():
        val_pred = es_model(X_val)
        val_loss = criterion(val_pred, y_val).item()
    es_val_losses.append(val_loss)

    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(es_model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            stop_epoch = epoch + 1
            print(f"Early stopping at epoch {stop_epoch}")
            break

# Restore best model
es_model.load_state_dict(best_model_state)
es_model.eval()
with torch.no_grad():
    test_pred = es_model(X_test_cnn.to(device)).cpu()
    es_acc = ((test_pred > 0.5).float() == y_test_cnn).float().mean()

print(f"Test accuracy with early stopping: {es_acc:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(es_train_losses, label='Training loss', linewidth=2)
plt.plot(es_val_losses, label='Validation loss', linewidth=2)
if stop_epoch < 200:
    best_epoch = stop_epoch - patience
    plt.axvline(best_epoch, color='red', linestyle='--', label=f'Best model (epoch {best_epoch})')
    plt.axvline(stop_epoch - 1, color='gray', linestyle=':', label=f'Early stop (epoch {stop_epoch})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Early Stopping')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Exercise 2: CNN Architecture Comparison

Compare different CNN architectures for sequence classification:

1. **Shallow CNN:** 1 convolutional layer (32 filters, kernel=12), global max pooling, dense output
2. **Deep CNN:** 3 convolutional layers (32 -> 64 -> 128 filters), with batch normalization after each
3. **Multi-scale CNN:** 3 parallel convolutional layers with different kernel sizes (4, 8, 16) concatenated

Train each for 50 epochs and compare test accuracy, number of parameters, and training time.

In [ ]:
# Solution: Exercise 2
import time

class ShallowCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(4, 32, kernel_size=12, padding=5)
        self.classifier = nn.Sequential(
            nn.Linear(32, 1), nn.Sigmoid()
        )

    def forward(self, x):
        x = torch.relu(self.conv(x))
        x = x.max(dim=2)[0]  # global max pooling
        return self.classifier(x)


class DeepCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(4, 32, kernel_size=8, padding=3),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=6, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=4, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):
        x = self.layers(x)
        x = x.mean(dim=2)
        return self.classifier(x)


class MultiScaleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv4 = nn.Conv1d(4, 32, kernel_size=4, padding=1)
        self.conv8 = nn.Conv1d(4, 32, kernel_size=8, padding=3)
        self.conv16 = nn.Conv1d(4, 32, kernel_size=16, padding=7)
        self.classifier = nn.Sequential(
            nn.Linear(96, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):
        c4 = torch.relu(self.conv4(x)).max(dim=2)[0]
        c8 = torch.relu(self.conv8(x)).max(dim=2)[0]
        c16 = torch.relu(self.conv16(x)).max(dim=2)[0]
        combined = torch.cat([c4, c8, c16], dim=1)
        return self.classifier(combined)


def train_and_evaluate(model_class, name, train_dl, X_test, y_test, epochs=50):
    model = model_class().to(device)
    opt = optim.Adam(model.parameters(), lr=0.001)
    n_params = sum(p.numel() for p in model.parameters())

    start = time.time()
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
    elapsed = time.time() - start

    model.eval()
    with torch.no_grad():
        preds = model(X_test.to(device)).cpu()
        acc = ((preds > 0.5).float() == y_test).float().mean().item()

    return {'Model': name, 'Parameters': f"{n_params:,}", 'Accuracy': f"{acc:.4f}",
            'Time (s)': f"{elapsed:.1f}"}


results_list = [
    train_and_evaluate(ShallowCNN, 'Shallow CNN', train_dl, X_test_cnn, y_test_cnn),
    train_and_evaluate(DeepCNN, 'Deep CNN + BatchNorm', train_dl, X_test_cnn, y_test_cnn),
    train_and_evaluate(MultiScaleCNN, 'Multi-scale CNN', train_dl, X_test_cnn, y_test_cnn),
]

print(pd.DataFrame(results_list).to_string(index=False))

### Exercise 3: Protein Sequence Autoencoder

Build an autoencoder for **protein amino acid composition**:

1. Generate 1000 synthetic proteins from 3 functional classes with distinct amino acid biases:
   - Membrane proteins: enriched in hydrophobic residues (A, V, L, I, F, W)
   - DNA-binding proteins: enriched in positively charged residues (K, R, H)
   - Enzymes: enriched in catalytic residues (H, C, D, E, S)
2. Compute 20-dimensional amino acid composition vectors
3. Train a standard autoencoder (20 -> 64 -> 2 -> 64 -> 20) with MSE loss
4. Visualize the 2D latent space colored by protein class
5. How well does the autoencoder separate the classes without ever seeing labels?

In [ ]:
# Solution: Exercise 3
amino_acids = list('ACDEFGHIKLMNPQRSTVWY')
aa_to_idx = {aa: i for i, aa in enumerate(amino_acids)}

def generate_protein(length, enriched_aas, enrichment=0.4):
    probs = np.ones(20) * (1 - enrichment) / (20 - len(enriched_aas))
    for aa in enriched_aas:
        probs[aa_to_idx[aa]] = enrichment / len(enriched_aas)
    probs /= probs.sum()
    return ''.join(np.random.choice(amino_acids, size=length, p=probs))


def aa_composition(seq):
    comp = np.zeros(20, dtype=np.float32)
    for aa in seq:
        if aa in aa_to_idx:
            comp[aa_to_idx[aa]] += 1
    return comp / len(seq)


# Generate proteins
protein_data = []
protein_labels = []
classes = [
    ('Membrane', list('AVLIFW')),
    ('DNA-binding', list('KRH')),
    ('Enzyme', list('HCDES')),
]

for label, (name, enriched) in enumerate(classes):
    for _ in range(333):
        seq = generate_protein(200, enriched)
        protein_data.append(aa_composition(seq))
        protein_labels.append(label)

X_prot = np.array(protein_data)
y_prot = np.array(protein_labels)

# Simple autoencoder with 2D bottleneck
class ProteinAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(20, 64), nn.ReLU(),
            nn.Linear(64, 2)  # 2D bottleneck for visualization
        )
        self.decoder = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(),
            nn.Linear(64, 20)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


X_prot_t = torch.FloatTensor(X_prot)
prot_ds = TensorDataset(X_prot_t)
prot_dl = DataLoader(prot_ds, batch_size=64, shuffle=True)

ae = ProteinAE().to(device)
optimizer = optim.Adam(ae.parameters(), lr=0.001)

for epoch in range(200):
    ae.train()
    for (batch,) in prot_dl:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon, _ = ae(batch)
        loss = nn.functional.mse_loss(recon, batch)
        loss.backward()
        optimizer.step()

# Extract latent representations
ae.eval()
with torch.no_grad():
    _, latent_prot = ae(X_prot_t.to(device))
    latent_prot = latent_prot.cpu().numpy()

plt.figure(figsize=(8, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71']
for i, (name, _) in enumerate(classes):
    mask = y_prot == i
    plt.scatter(latent_prot[mask, 0], latent_prot[mask, 1],
                c=colors[i], label=name, alpha=0.6, s=20)

plt.xlabel('Latent dimension 1')
plt.ylabel('Latent dimension 2')
plt.title('Autoencoder Latent Space -- Unsupervised Protein Class Separation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Exercise 4: Sequence-Level Attention Visualization

Build a simple attention-based classifier and visualize what the model attends to:

1. Create a model with: embedding -> single attention layer -> classification head
2. Train it on the promoter dataset
3. For a correctly classified promoter, extract and plot the attention weights as a heatmap over the sequence positions
4. Do the highest attention positions correspond to known motif regions (around position 80--90 where we placed TATA boxes)?

In [ ]:
# Solution: Exercise 4
class AttentionClassifier(nn.Module):
    """Classifier with explicit attention layer for interpretability."""

    def __init__(self, input_dim=4, d_model=32):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        # Single-head attention for interpretability
        self.attention = nn.MultiheadAttention(d_model, num_heads=1, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
        self._attn_weights = None

    def forward(self, x):
        # x: (batch, 4, seq_length) -> (batch, seq_length, 4)
        x = x.transpose(1, 2)
        x = self.proj(x)  # (batch, seq_length, d_model)
        attn_out, attn_weights = self.attention(x, x, x, need_weights=True)
        self._attn_weights = attn_weights.detach()
        # Pool over positions
        pooled = attn_out.mean(dim=1)
        return self.classifier(pooled)


attn_model = AttentionClassifier().to(device)
optimizer = optim.Adam(attn_model.parameters(), lr=0.001)

for epoch in range(50):
    attn_model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(attn_model(xb), yb)
        loss.backward()
        optimizer.step()

# Get attention for a promoter sequence
attn_model.eval()
promoter_example = X_encoded[promoter_idx:promoter_idx+1].to(device)
with torch.no_grad():
    pred = attn_model(promoter_example)
    attn = attn_model._attn_weights.cpu().numpy()[0]  # (seq_length, seq_length)

# Average attention received by each position (column mean)
pos_attention = attn.mean(axis=0)

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# Attention per position
axes[0].bar(range(len(pos_attention)), pos_attention, width=1.0, color='steelblue', alpha=0.7)
axes[0].axvspan(80, 90, alpha=0.2, color='red', label='Expected TATA region')
axes[0].set_xlabel('Sequence Position')
axes[0].set_ylabel('Mean Attention')
axes[0].set_title(f'Attention per Position (predicted: {pred.item():.3f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Attention heatmap (first 50 positions for readability)
axes[1].imshow(attn[:50, :50], cmap='viridis', aspect='auto')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Query position')
axes[1].set_title('Attention Heatmap (first 50 positions)')

plt.tight_layout()
plt.show()

### Exercise 5: VAE for Molecule Latent Space

Build a VAE for molecular fingerprints:

1. Simulate 1500 molecules from 3 drug classes with different structural fingerprint patterns:
   - Kinase inhibitors: bits 0--30 active
   - GPCR ligands: bits 30--60 active
   - Ion channel blockers: bits 60--90 active
   (Use 256-bit binary fingerprints with noise)
2. Train a VAE with latent_dim=2
3. Visualize the latent space colored by drug class
4. Generate 50 new "molecules" by sampling from the latent space: `z ~ N(0, I)`
5. Decode the samples and check if they resemble real fingerprints (plot a few)

In [ ]:
# Solution: Exercise 5
n_bits = 256
n_per_class = 500

fingerprints = []
drug_classes = []

class_profiles = [
    ('Kinase inhibitors', range(0, 30)),
    ('GPCR ligands', range(30, 60)),
    ('Ion channel blockers', range(60, 90)),
]

for label, (name, active_bits) in enumerate(class_profiles):
    for _ in range(n_per_class):
        fp = np.random.binomial(1, 0.05, n_bits).astype(np.float32)  # sparse background
        for bit in active_bits:
            fp[bit] = np.random.binomial(1, 0.7)  # 70% chance each class-specific bit is on
        fingerprints.append(fp)
        drug_classes.append(label)

X_mol = np.array(fingerprints)
y_mol = np.array(drug_classes)

# Train VAE
mol_vae = VAE(input_dim=n_bits, hidden_dim=128, latent_dim=2).to(device)
optimizer = optim.Adam(mol_vae.parameters(), lr=0.001)

X_mol_t = torch.FloatTensor(X_mol)
mol_ds = TensorDataset(X_mol_t)
mol_dl = DataLoader(mol_ds, batch_size=64, shuffle=True)

for epoch in range(150):
    mol_vae.train()
    for (batch,) in mol_dl:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = mol_vae(batch)
        loss = vae_loss(recon, batch, mu, logvar, beta=0.1)
        loss.backward()
        optimizer.step()

# Visualize latent space
mol_vae.eval()
with torch.no_grad():
    mu_mol, _ = mol_vae.encode(X_mol_t.to(device))
    latent_mol = mu_mol.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Latent space
colors = ['#e74c3c', '#3498db', '#2ecc71']
for i, (name, _) in enumerate(class_profiles):
    mask = y_mol == i
    axes[0].scatter(latent_mol[mask, 0], latent_mol[mask, 1],
                    c=colors[i], label=name, alpha=0.5, s=15)
axes[0].set_title('VAE Latent Space -- Molecular Fingerprints')
axes[0].set_xlabel('z1')
axes[0].set_ylabel('z2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Generate new molecules
with torch.no_grad():
    z_samples = torch.randn(50, 2).to(device)
    generated = mol_vae.decode(z_samples).cpu().numpy()

# Show 5 generated fingerprints vs 5 real ones
comparison = np.vstack([X_mol[:5], (generated[:5] > 0.5).astype(float)])
axes[1].imshow(comparison, aspect='auto', cmap='Blues', interpolation='nearest')
axes[1].set_yticks(range(10))
axes[1].set_yticklabels([f'Real {i}' for i in range(5)] + [f'Generated {i}' for i in range(5)])
axes[1].set_xlabel('Fingerprint bit')
axes[1].set_title('Real vs Generated Molecular Fingerprints')

plt.tight_layout()
plt.show()

# Check that generated molecules have structure
gen_binary = (generated > 0.5).astype(float)
print(f"Mean bits active (real):      {X_mol.mean():.3f}")
print(f"Mean bits active (generated): {gen_binary.mean():.3f}")

### Exercise 6: Transfer Learning Simulation

Simulate transfer learning for a biological task:

1. **Pre-train** a CNN on 5000 sequences for a "general" task (GC-content classification: high-GC vs low-GC)
2. **Fine-tune** the pre-trained model on only 100 sequences for a "specific" task (promoter classification)
3. Compare against training from scratch on the same 100 promoter sequences
4. Try two fine-tuning strategies:
   - Freeze convolutional layers, only train the classifier head
   - Fine-tune all layers with a reduced learning rate
5. Report accuracy for all three approaches

In [ ]:
# Solution: Exercise 6

# 1. Generate pre-training data: GC-content classification
def generate_gc_data(n=5000, seq_length=200):
    seqs, labs = [], []
    for _ in range(n):
        if np.random.random() < 0.5:
            # High GC
            seq = ''.join(np.random.choice(['A','T','G','C'], size=seq_length, p=[0.15, 0.15, 0.35, 0.35]))
            labs.append(1)
        else:
            # Low GC
            seq = ''.join(np.random.choice(['A','T','G','C'], size=seq_length, p=[0.35, 0.35, 0.15, 0.15]))
            labs.append(0)
        seqs.append(seq)
    return seqs, np.array(labs)


gc_seqs, gc_labels = generate_gc_data(5000)
gc_encoded = one_hot_encode(gc_seqs)
gc_labels_t = torch.FloatTensor(gc_labels).unsqueeze(1)

gc_ds = TensorDataset(gc_encoded, gc_labels_t)
gc_dl = DataLoader(gc_ds, batch_size=64, shuffle=True)

# Pre-train
pretrained_cnn = SequenceCNN().to(device)
optimizer = optim.Adam(pretrained_cnn.parameters(), lr=0.001)
for epoch in range(20):
    pretrained_cnn.train()
    for xb, yb in gc_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(pretrained_cnn(xb), yb)
        loss.backward()
        optimizer.step()
print("Pre-training complete.")

# 2. Small promoter dataset (100 samples)
small_promo_idx = np.concatenate([np.where(labels == 1)[0][:50], np.where(labels == 0)[0][:50]])
np.random.shuffle(small_promo_idx)
sp_train_idx, sp_test_idx = small_promo_idx[:80], small_promo_idx[80:]

sp_train_ds = TensorDataset(X_encoded[sp_train_idx], y_tensor[sp_train_idx])
sp_train_dl = DataLoader(sp_train_ds, batch_size=16, shuffle=True)


def fine_tune(model, freeze_conv, lr, name):
    model = copy.deepcopy(model).to(device)
    if freeze_conv:
        for param in model.conv_layers.parameters():
            param.requires_grad = False
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    for epoch in range(50):
        model.train()
        for xb, yb in sp_train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        preds = model(X_encoded[sp_test_idx].to(device)).cpu()
        acc = ((preds > 0.5).float() == y_tensor[sp_test_idx]).float().mean().item()
    return {'Strategy': name, 'Accuracy': f"{acc:.4f}"}


# 3. From scratch
scratch_result = fine_tune(SequenceCNN(), freeze_conv=False, lr=0.001, name='From scratch')

# 4a. Transfer: freeze conv layers
frozen_result = fine_tune(pretrained_cnn, freeze_conv=True, lr=0.001, name='Transfer (frozen conv)')

# 4b. Transfer: fine-tune all with lower LR
finetune_result = fine_tune(pretrained_cnn, freeze_conv=False, lr=0.0001, name='Transfer (full fine-tune)')

print(pd.DataFrame([scratch_result, frozen_result, finetune_result]).to_string(index=False))

---
## Key Takeaways

1. **Choose the right architecture for the task:** CNNs for local motif detection, LSTMs/Transformers for long-range dependencies, VAEs for dimensionality reduction and generation.

2. **Classical ML is not obsolete:** For tabular biological data with limited samples, random forests and SVMs often outperform deep learning. Use the right tool for the job.

3. **Pre-trained models are transformative:** ESM-2, AlphaFold, and DNABERT capture biological knowledge from massive datasets. Transfer learning lets you leverage this with minimal data.

4. **Overfitting is the central challenge:** Biological datasets are often small relative to model capacity. Employ early stopping, dropout, weight decay, data augmentation, and cross-validation.

5. **Interpretability matters:** Saliency maps, attention weights, and learned filters help connect model predictions to biological mechanisms.

6. **Start simple, add complexity only when needed:** Begin with a feedforward network on hand-crafted features. Move to CNNs on raw sequences. Consider transformers only if justified by data size and task complexity.

**Recommended next steps:**
- Explore the [ESM](https://github.com/facebookresearch/esm) repository for protein language models
- Try [scvi-tools](https://scvi-tools.org/) for single-cell analysis with deep learning
- Read the [AlphaFold2 paper](https://doi.org/10.1038/s41586-021-03819-2) to understand the Evoformer architecture
- Experiment on [Kaggle Bioinformatics competitions](https://www.kaggle.com/competitions) with real datasets

---

[← Previous: Molecular Modeling and Docking](../../Tier_3_Applied_Bioinformatics/09_Molecular_Modeling_and_Docking/01_molecular_modeling_and_docking.ipynb) | [Next: Clinical Genomics: From Variants to Patient Care →](../../Tier_3_Applied_Bioinformatics/11_Clinical_Genomics/01_clinical_genomics.ipynb)